# Reduced Model Comparison

Compares four variants of the hierarchical active inference model:

| Model | Modules | Description |
|-------|---------|-------------|
| **Full (M+T+D)** | M, T, D | Imported from `sim.py` |
| **No-D (M+T)** | M, T | T still identifies threat and triggers approach; no danger/flee switching |
| **No-T (M+D)** | M, D | D reacts to raw proximity (always assumes threat present); no identity confirmation |
| **M only** | M | Motor agent with fixed preferences; no higher-level control |

All models share the same parameter signature and arena.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import entropy as scipy_entropy

from sim import run_sim          # Full model (M+T+D)
from setup import setup
from utils import world_env, decay_qs, calculate_metrics, final_mask, active_cells

## Reduced simulation functions

Each function is a stripped-down variant of `sim.py:run_sim`. All share the same
parameter signature so they can be swapped in/out directly.

In [2]:
# ─── No-D model (M + T only) ────────────────────────────────────────────────
# T still fires every T_ticks and can switch M to APPROACH mode.
# There is no danger/flee switching — D's cooldown and DANGER C-scaling are absent.
# M's preferences are modulated only by T (approach or default), never by a flee signal.

def run_sim_no_D(
    max_steps=2000, id_threshold=0.8,
    M_fr=0.1, T_fr=0.2,
    T_ticks=16, D_ticks=48,           # D_ticks kept for signature parity but unused
    k_shelter=0.6, k_threat=0.8,
    threat_grad=[-0.10,-0.10,-0.10,-0.10,-0.10,-0.12,-0.15,-0.18,-0.21,-0.25],
    shelter_grad=[-0.0, 3.0],
    delta_stay=0.15, epistemic_drive=1.0,
    T_scale=(-3.0, -3.0), D_scale=(5.0, 20.0),
    sensory_imprecision=0.75, printing=False,
):
    arena, build_scaled_C, M_agent, _D_agent, T_agent, D_control_scales, T_control_scales, \
        *_ , rightcol_states, leftcol_states = setup(
            k_shelter=k_shelter, k_threat=k_threat,
            threat_grad=threat_grad, shelter_grad=shelter_grad,
            delta_stay=delta_stay, epistemic_drive=epistemic_drive,
            T_action=T_scale, D_action=D_scale,
            sensory_imprecision=sensory_imprecision, printing=False,
        )

    world = world_env(arena=arena, true_agent_pos=(1, 0),
                      true_threat_pos=rightcol_states[0],
                      true_shelter_pos=np.array(leftcol_states, dtype=int))

    agent_obs, threat_obs, shelter_obs = world.start()
    obs = [threat_obs, shelter_obs, agent_obs]

    base_scale   = D_control_scales[0]
    current_scale = base_scale
    T_ticker     = T_ticks

    history = {'agent_loc': [], 'M_action': [], 'context': [],
               'T_beliefs': [], 'T_action': [], 'T_act_t': []}

    for t in range(max_steps):
        M_qs = M_agent.infer_states(obs)
        M_agent.infer_policies()
        M_action = M_agent.sample_action()[0]

        current_state = world.agent_pos
        agent_obs, threat_obs, shelter_obs = world.step(M_action)
        obs = [threat_obs, shelter_obs, agent_obs]

        # T level: threat identification
        T_qs = T_agent.infer_states([obs[0]])

        # T fires every T_ticks — can only switch M to APPROACH or back to DEFAULT
        if T_ticker == 0:
            T_ticker += T_ticks + 1
            T_qpi, T_G = T_agent.infer_policies()
            T_action = int(T_agent.sample_action()[0])
            history['T_act_t'].append(t)
            history['T_action'].append(T_action)
            updated_scale = T_control_scales[T_action]
            if updated_scale != current_scale:
                label = 'DEFAULT' if updated_scale == base_scale else 'APPROACH'
                if printing: print(f't={t} | T → {label}')
                M_agent.C = build_scaled_C(updated_scale)
                current_scale = updated_scale

        T_ticker -= 1

        # Decay
        M_agent.reset(init_qs=np.array(
            [M_qs[0], decay_qs(M_qs[1], M_fr), M_qs[2]], dtype=object))
        T_agent.reset(init_qs=np.array(
            [decay_qs(T_qs[0], T_fr), T_qs[1]], dtype=object))

        ctx = 'DEFAULT' if current_scale == base_scale else 'APPROACH'
        history['agent_loc'].append(current_state)
        history['M_action'].append(int(M_action))
        history['context'].append(ctx)
        history['T_beliefs'].append(T_qs)

    return history

In [3]:
# ─── No-T model (M + D only) ────────────────────────────────────────────────
# T's threat-identity inference is removed. D now uses raw spatial proximity
# (from M's beliefs about threat location) and always treats the object as a
# threat (D_id_obs = 1 always → D_obs in range 10-19 of the joint obs space).
# D fires periodically (D_ticks) or immediately when d_to_cage < distance_threshold.

def run_sim_no_T(
    max_steps=2000, id_threshold=0.8,
    M_fr=0.1, D_fr=0.2,
    T_ticks=16, D_ticks=48,           # T_ticks kept for signature parity but unused
    k_shelter=0.6, k_threat=0.8,
    threat_grad=[-0.10,-0.10,-0.10,-0.10,-0.10,-0.12,-0.15,-0.18,-0.21,-0.25],
    shelter_grad=[-0.0, 3.0],
    delta_stay=0.15, epistemic_drive=1.0,
    T_scale=(-3.0, -3.0), D_scale=(5.0, 20.0),
    sensory_imprecision=0.75, printing=False,
):
    arena, build_scaled_C, M_agent, D_agent, _T_agent, D_control_scales, T_control_scales, \
        *_ , rightcol_states, leftcol_states = setup(
            k_shelter=k_shelter, k_threat=k_threat,
            threat_grad=threat_grad, shelter_grad=shelter_grad,
            delta_stay=delta_stay, epistemic_drive=epistemic_drive,
            T_action=T_scale, D_action=D_scale,
            sensory_imprecision=sensory_imprecision, printing=False,
        )

    world = world_env(arena=arena, true_agent_pos=(1, 0),
                      true_threat_pos=rightcol_states[0],
                      true_shelter_pos=np.array(leftcol_states, dtype=int))

    agent_obs, threat_obs, shelter_obs = world.start()
    obs = [threat_obs, shelter_obs, agent_obs]

    base_scale    = D_control_scales[0]
    current_scale = base_scale
    D_ticker      = D_ticks
    D_cooldown    = False
    D_DISTANCE_THRESHOLD = 3  # mirrors setup.py distance_threshold

    history = {'agent_loc': [], 'M_action': [], 'context': [],
               'D_beliefs': [], 'D_action': [], 'D_act_t': []}

    for t in range(max_steps):
        M_qs = M_agent.infer_states(obs)
        M_agent.infer_policies()
        M_action = M_agent.sample_action()[0]

        current_state = world.agent_pos
        agent_obs, threat_obs, shelter_obs = world.step(M_action)
        obs = [threat_obs, shelter_obs, agent_obs]

        # D level: compute distance from M's belief about agent and threat positions
        mp_A_rc = arena.state_idx_to_rc(np.argmax(M_qs[0]))
        mp_T_rc = arena.state_idx_to_rc(np.argmax(M_qs[1]))
        threat_cluster = [
            (mp_T_rc[0],   mp_T_rc[1]),
            (mp_T_rc[0],   mp_T_rc[1]-1),
            (mp_T_rc[0]-1, mp_T_rc[1]),
            (mp_T_rc[0]-1, mp_T_rc[1]-1),
        ]
        d_to_cage = min(
            abs(mp_A_rc[0]-r) + abs(mp_A_rc[1]-c)
            for r, c in threat_cluster
            if 0 <= r < arena.rows and 0 <= c < arena.cols
        )

        # No T agent: always assume threat is real (D_id_obs = 1)
        D_obs = min(d_to_cage, 9) + 10
        D_qs  = D_agent.infer_states([D_obs])

        # D fires periodically OR when proximity < threshold (raw proximity trigger)
        proximity_trigger = (d_to_cage < D_DISTANCE_THRESHOLD) and not D_cooldown
        if (D_ticker == 0) or proximity_trigger:
            D_qpi, D_G = D_agent.infer_policies()
            D_action = int(D_agent.sample_action()[0])
            history['D_act_t'].append(t)
            history['D_action'].append(D_action)
            updated_scale = D_control_scales[D_action]
            if updated_scale != current_scale:
                D_scale_name = 'SAFE' if updated_scale == base_scale else 'DANGER'
                if D_scale_name == 'DANGER':
                    if printing: print(f't={t} | D → DANGER')
                    M_agent.C = build_scaled_C(updated_scale)
                    current_scale = updated_scale
                    D_ticker += D_ticks // 2
                    D_cooldown = True
            if D_ticker == 0:
                D_ticker += D_ticks + 1
                if D_cooldown:
                    D_cooldown = False

        D_ticker -= 1

        # Decay
        M_agent.reset(init_qs=np.array(
            [M_qs[0], decay_qs(M_qs[1], M_fr), M_qs[2]], dtype=object))
        D_agent.reset(init_qs=np.array(
            [decay_qs(D_qs[0], D_fr)], dtype=object))

        ctx = 'SAFE' if current_scale == base_scale else 'DANGER'
        history['agent_loc'].append(current_state)
        history['M_action'].append(int(M_action))
        history['context'].append(ctx)
        history['D_beliefs'].append(D_qs)

    return history

In [4]:
# ─── M-only model ────────────────────────────────────────────────────────────
# Motor agent alone with fixed preferences throughout.
# No T, no D — no higher-level context switching of any kind.

def run_sim_M_only(
    max_steps=2000, id_threshold=0.8,
    M_fr=0.1, D_fr=0.2, T_fr=0.2,
    T_ticks=16, D_ticks=48,
    k_shelter=0.6, k_threat=0.8,
    threat_grad=[-0.10,-0.10,-0.10,-0.10,-0.10,-0.12,-0.15,-0.18,-0.21,-0.25],
    shelter_grad=[-0.0, 3.0],
    delta_stay=0.15, epistemic_drive=1.0,
    T_scale=(-3.0, -3.0), D_scale=(5.0, 20.0),
    sensory_imprecision=0.75, printing=False,
):
    arena, build_scaled_C, M_agent, *_, rightcol_states, leftcol_states = setup(
        k_shelter=k_shelter, k_threat=k_threat,
        threat_grad=threat_grad, shelter_grad=shelter_grad,
        delta_stay=delta_stay, epistemic_drive=epistemic_drive,
        T_action=T_scale, D_action=D_scale,
        sensory_imprecision=sensory_imprecision, printing=False,
    )

    world = world_env(arena=arena, true_agent_pos=(1, 0),
                      true_threat_pos=rightcol_states[0],
                      true_shelter_pos=np.array(leftcol_states, dtype=int))

    agent_obs, threat_obs, shelter_obs = world.start()
    obs = [threat_obs, shelter_obs, agent_obs]

    history = {'agent_loc': [], 'M_action': [], 'context': []}

    for t in range(max_steps):
        M_qs = M_agent.infer_states(obs)
        M_agent.infer_policies()
        M_action = M_agent.sample_action()[0]

        current_state = world.agent_pos
        agent_obs, threat_obs, shelter_obs = world.step(M_action)
        obs = [threat_obs, shelter_obs, agent_obs]

        # Decay M beliefs only
        M_agent.reset(init_qs=np.array(
            [M_qs[0], decay_qs(M_qs[1], M_fr), M_qs[2]], dtype=object))

        history['agent_loc'].append(current_state)
        history['M_action'].append(int(M_action))
        history['context'].append('FIXED')

    return history

## Parameters

Shared parameter set for all models. Edit `PARAMS` to test different phenotypes.

In [ ]:
# --- Default comparison params (edit here) ---
PARAMS = dict(
    max_steps         = 2000,
    k_shelter         = 3.5,
    k_threat          = 0.4,
    id_threshold      = 0.2,
    sensory_imprecision = 0.12,
    delta_stay        = 3.0,
    epistemic_drive   = 1.0,
)

# Optional: define multiple phenotype param sets for side-by-side comparison
PARAM_SETS = {
    'Default': PARAMS,
    # Uncomment and fill in with best-fit params from lowest_losses_full_adj.csv:
    # 'Resilient (14_def2)': dict(max_steps=2000, k_shelter=..., k_threat=..., ...),
    # 'Susceptible (16_def2)': dict(max_steps=2000, k_shelter=..., k_threat=..., ...),
}

print('Running with params:')
for k, v in PARAMS.items():
    print(f'  {k}: {v}')

## Run all four models

In [ ]:
# --- Default comparison params (edit here) ---
# NOTE: max_steps=100 for quick verification — change to 2000 for real analysis
PARAMS = dict(
    max_steps         = 100,
    k_shelter         = 3.5,
    k_threat          = 0.4,
    id_threshold      = 0.2,
    sensory_imprecision = 0.12,
    delta_stay        = 3.0,
    epistemic_drive   = 1.0,
)

# Optional: define multiple phenotype param sets for side-by-side comparison
PARAM_SETS = {
    'Default': PARAMS,
    # Uncomment and fill in with best-fit params from lowest_losses_full_adj.csv:
    # 'Resilient (14_def2)': dict(max_steps=2000, k_shelter=..., k_threat=..., ...),
    # 'Susceptible (16_def2)': dict(max_steps=2000, k_shelter=..., k_threat=..., ...),
}

print('Running with params:')
for k, v in PARAMS.items():
    print(f'  {k}: {v}')

## Behavioral metrics comparison

In [ ]:
metrics_rows = {}
for name, h in results.items():
    traj = pd.DataFrame({'location': h['agent_loc']})
    metrics_rows[name] = calculate_metrics(traj)

metrics_df = pd.DataFrame(metrics_rows).T
metrics_df.index.name = 'Model'

# Round for readability
display_df = metrics_df.copy()
for col in ['t_shelter', 't_investigating', 'entropy', 'laziness']:
    display_df[col] = display_df[col].round(3)
for col in ['n_sh_co', 'n_co_sh', 'n_co_ch', 'n_ch_co']:
    display_df[col] = display_df[col].astype(int)

print(display_df.to_string())

## Spatial occupancy heatmaps

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 3.5))

# Build occupancy grid for each model
n_active = active_cells.shape[0]  # 57

for ax, (name, h) in zip(axes, results.items()):
    # Count visits per state
    counts = np.zeros(n_active)
    for loc in h['agent_loc']:
        if 0 <= loc < n_active:
            counts[loc] += 1
    counts /= counts.sum() + 1e-9

    # Map back to 2D grid
    grid = np.full(final_mask.shape, np.nan)
    for state_idx, (r, c) in enumerate(active_cells):
        grid[r, c] = counts[state_idx]

    im = ax.imshow(grid, cmap='hot_r', aspect='auto',
                   vmin=0, vmax=counts.max())
    plt.colorbar(im, ax=ax, fraction=0.04, label='P(visit)')

    # Mark shelter (leftmost col) and threat cage
    for r in range(final_mask.shape[0]):
        if final_mask[r, 0]:
            ax.add_patch(plt.Rectangle((-.5, r-.5), 1, 1,
                         fill=False, edgecolor='lime', lw=2))
    for r, c in [(3,13),(3,14),(4,13),(4,14)]:
        if final_mask[r, c]:
            ax.add_patch(plt.Rectangle((c-.5, r-.5), 1, 1,
                         fill=False, edgecolor='cyan', lw=2))

    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')

fig.suptitle('Spatial occupancy (green = shelter, cyan = threat cage)', fontsize=12)
plt.tight_layout()
plt.savefig('model_comparison_heatmaps.svg', bbox_inches='tight')
plt.show()
print('Saved model_comparison_heatmaps.svg')

## Zone membership over time

In [ ]:
def get_zone(state_idx):
    """Map a state index to Shelter / Corridor / Chamber."""
    r, c = active_cells[state_idx]
    if c == 0:       return 'Shelter'
    elif c >= 9:     return 'Chamber'
    else:            return 'Corridor'

ZONE_COLORS = {'Shelter': '#2196F3', 'Corridor': '#FF9800', 'Chamber': '#F44336'}
ZONES = ['Shelter', 'Corridor', 'Chamber']

# Rolling window for smoothed occupancy fraction
WINDOW = 100

fig, axes = plt.subplots(n_models, 1, figsize=(14, 2.5 * n_models), sharex=True)

for ax, (name, h) in zip(axes, results.items()):
    zones_series = [get_zone(loc) for loc in h['agent_loc']]
    zone_df = pd.DataFrame({'zone': zones_series})
    dummies  = pd.get_dummies(zone_df['zone']).reindex(columns=ZONES, fill_value=0)
    smoothed = dummies.rolling(WINDOW, min_periods=1).mean()

    bottom = np.zeros(len(smoothed))
    for zone in ZONES:
        if zone in smoothed.columns:
            ax.fill_between(range(len(smoothed)),
                            bottom, bottom + smoothed[zone].values,
                            color=ZONE_COLORS[zone], alpha=0.75, label=zone)
            bottom += smoothed[zone].values

    # Overlay context switches (if available)
    if 'context' in h:
        ctx = h['context']
        for t in range(1, len(ctx)):
            if ctx[t] != ctx[t-1]:
                color = 'red' if 'DANGER' in ctx[t] else ('green' if 'APPROACH' in ctx[t] else 'gray')
                ax.axvline(t, color=color, alpha=0.5, lw=0.8)

    ax.set_ylim(0, 1)
    ax.set_ylabel('Zone fraction', fontsize=9)
    ax.set_title(name, fontsize=10)
    ax.legend(loc='upper right', fontsize=8, ncol=3)

axes[-1].set_xlabel('Timestep')
fig.suptitle(f'Zone occupancy over time (smoothed, window={WINDOW})\n'
             'Vertical lines: context switches (red=DANGER, green=APPROACH, gray=DEFAULT)',
             fontsize=11)
plt.tight_layout()
plt.savefig('model_comparison_zones.svg', bbox_inches='tight')
plt.show()
print('Saved model_comparison_zones.svg')

## Context switch timeline

In [ ]:
# Show when each model's context changed (D switches for Full/No-T, T switches for No-D)
fig, axes = plt.subplots(n_models, 1, figsize=(14, 1.6 * n_models), sharex=True)

CTX_COLOR = {'SAFE': '#4CAF50', 'DANGER': '#F44336',
             'DEFAULT': '#9E9E9E', 'APPROACH': '#FF9800', 'FIXED': '#607D8B'}

for ax, (name, h) in zip(axes, results.items()):
    ctx = h.get('context', ['FIXED'] * len(h['agent_loc']))
    # Build segments
    segments = []
    start = 0
    for i in range(1, len(ctx)):
        if ctx[i] != ctx[i-1]:
            segments.append((start, i, ctx[i-1]))
            start = i
    segments.append((start, len(ctx), ctx[-1]))

    for s, e, label in segments:
        ax.barh(0, e - s, left=s, height=0.8,
                color=CTX_COLOR.get(label, 'purple'), alpha=0.9)

    # Legend patches
    from matplotlib.patches import Patch
    unique_labels = list(dict.fromkeys(c for _, _, c in segments))
    handles = [Patch(color=CTX_COLOR.get(l, 'purple'), label=l) for l in unique_labels]
    ax.legend(handles=handles, loc='upper right', fontsize=8, ncol=len(unique_labels))
    ax.set_yticks([])
    ax.set_title(name, fontsize=9)

axes[-1].set_xlabel('Timestep')
fig.suptitle('Context mode timeline', fontsize=11)
plt.tight_layout()
plt.savefig('model_comparison_context.svg', bbox_inches='tight')
plt.show()
print('Saved model_comparison_context.svg')

## Metrics bar chart comparison

In [ ]:
metric_display = {
    't_shelter':       'Time in shelter',
    't_investigating': 'Time investigating',
    'n_sh_co':         'Shelter→Corridor',
    'n_co_sh':         'Corridor→Shelter',
    'n_co_ch':         'Corridor→Chamber',
    'n_ch_co':         'Chamber→Corridor',
    'entropy':         'Spatial entropy',
    'laziness':        'Laziness (stay frac)',
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
colors = ['#2196F3', '#FF9800', '#F44336', '#4CAF50']
model_names = list(results.keys())

for ax, (metric_key, metric_label) in zip(axes, metric_display.items()):
    vals = [metrics_rows[m][metric_key] for m in model_names]
    bars = ax.bar(range(len(model_names)), vals, color=colors, alpha=0.85)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([m.replace(' (', '\n(') for m in model_names], fontsize=8)
    ax.set_title(metric_label, fontsize=10)
    ax.set_ylabel(metric_key, fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7)

fig.suptitle('Behavioral metrics across model variants', fontsize=13)
plt.tight_layout()
plt.savefig('model_comparison_metrics.svg', bbox_inches='tight')
plt.show()
print('Saved model_comparison_metrics.svg')

## Multi-parameter comparison (optional)

Run all four models across multiple parameter sets (e.g., different phenotype fits)
and compare. Edit `PARAM_SETS` at the top of the notebook to add parameter vectors.

In [ ]:
if len(PARAM_SETS) > 1:
    all_metrics = {}
    for pname, pdict in PARAM_SETS.items():
        all_metrics[pname] = {}
        for mname, fn in models.items():
            h = fn(**pdict)
            traj = pd.DataFrame({'location': h['agent_loc']})
            all_metrics[pname][mname] = calculate_metrics(traj)

    # Print summary table per metric
    for metric in ['t_shelter', 't_investigating', 'entropy']:
        print(f'\n--- {metric} ---')
        rows = {}
        for pname in PARAM_SETS:
            rows[pname] = {mname: round(all_metrics[pname][mname][metric], 3)
                           for mname in models}
        print(pd.DataFrame(rows).T.to_string())
else:
    print('Only one param set defined — edit PARAM_SETS to compare multiple phenotypes.')